Social Value Lancashire

Importing Libraries Below


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    


Settings Section

In [ ]:
EXCEL_FILE = "SocialValueData.xlsx"
CSV_DIR = "results_csv"
FIG_DIR = "figures"

os.makedirs(CSV_DIR, exist_ok=True)

# The 14 real Lancashire local authorities
CANONICAL_LAS = {
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",
}

#min years and LAs for forecast
MIN_LAS_FOR_FORECAST = 10
MIN_YEARS_FOR_FORECAST = 5
FORECAST_YEARS_AHEAD = 3

# IMD groups to us as predictors of crime
NON_CRIME_IMD_DOMAINS = ["Income", "Employment", "Health", "Education", "Barriers", "Living"]

CLUSTER_COLORS = ["#2A10BE", "#C715A0", "#1BD415", "#DB7413"]

#condensed list features for the correlation heatmap- too many illegible
HEATMAP_FEATURES = [
    "children_low_income_relative_level",
    "residual_waste_per_household_kg_level",
    "pct_waste_recycled_level",
    "anxiety_high_pct_level",
    "life_satisfaction_low_pct_level",
    "rough_sleeping_single_night_level",
    "post16_18_positive_destination_pct_level",
    "pct_adults_active_raw_level",
    "mean_imd_score",
    "imd_domain_Barriers",
    "imd_domain_Health",
    "imd_domain_Living",
]

HEATMAP_LABELS = {
    "gdhi_level": "GDHI (level)", "gdhi_trend": "GDHI (trend)",
    "children_low_income_relative_level": "Child. low income\n(level)",
    "children_low_income_relative_trend": "Child. low income\n(trend)",
    "residual_waste_per_household_kg_level": "Residual waste\n(level)",
    "residual_waste_per_household_kg_trend": "Residual waste\n(trend)",
    "pct_waste_recycled_level": "% waste\nrecycled (level)",
    "pct_waste_recycled_trend": "% waste\nrecycled (trend)",
    "anxiety_high_pct_level": "Anxiety, high\n(level)",
    "anxiety_high_pct_trend": "Anxiety, high\n(trend)",
    "life_satisfaction_low_pct_level": "Life satisf., low\n(level)",
    "life_satisfaction_low_pct_trend": "Life satisf., low\n(trend)",
    "rough_sleeping_single_night_level": "Rough sleeping,\nsingle night (level)",
    "rough_sleeping_single_night_trend": "Rough sleeping,\nsingle night (trend)",
    "post16_18_positive_destination_pct_level": "Post-16-18 positive\ndestination (level)",
    "post16_18_positive_destination_pct_trend": "Post-16-18 positive\ndestination (trend)",
    "pct_adults_active_raw_level": "% adults active,\nraw (level)",
    "pct_adults_active_raw_trend": "% adults active,\nraw (trend)",
    "mean_imd_score": "IMD (overall)",
    "imd_domain_Barriers": "IMD: Barriers",
    "imd_domain_Health": "IMD: Health",
    "imd_domain_Living": "IMD: Living Env.",
}

#Figure stuff
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})

Helper Code

In [ ]:
def clean_la_name(name):
    #Cleaning local authority names so consistent
    if pd.isna(name):
        return np.nan
    name = str(name).strip()
    for suffix in [" Borough Council", " City Council", " District Council", " Council", " LA", " District"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)].strip()
    return name


def load_sheet(xls, keyword):
    #find sheet
    matches = [s for s in xls.sheet_names if keyword.lower() in s.lower().replace(" ", "")]
    if not matches:
        raise ValueError(f"No sheet matching '{keyword}' found in the workbook.")
    print(f"  loading sheet '{matches[0]}'")
    return pd.read_excel(xls, sheet_name=matches[0])


1. Loading Workbook

In [ ]:
print(f"Looking for {EXCEL_FILE} in {os.getcwd()}")
if not os.path.exists(EXCEL_FILE):
    raise FileNotFoundError(f"'{EXCEL_FILE}' not found, check in the working directory and name correct")
xls = pd.ExcelFile(EXCEL_FILE)
print(f"Opened {EXCEL_FILE}: {len(xls.sheet_names)} sheets\n")

sheets = {
    "income": load_sheet(xls, "GrossDisposableIncome"),
    "children": load_sheet(xls, "ChildrenLowIncome"),
    "waste": load_sheet(xls, "WasteRecycled"),
    "crime": load_sheet(xls, "CrimeRates"),
    "imd": load_sheet(xls, "IMD"),
    "life": load_sheet(xls, "LifeSatisfaction"),
    "rough_sleeping": load_sheet(xls, "SleepingRough"),
    "post16_18": load_sheet(xls, "Post16-18Employment"),
    "active": load_sheet(xls, "AdultsActive"),
}

2. Tidy Sheets: Each raw sheet is turned into a tidy (local_authority,year,indicator, value) time series structure

In [ ]:
# income
df_income_ts = sheets["income"].copy()
df_income_ts["local_authority"] = df_income_ts["Region name"].apply(clean_la_name)
df_income_ts = df_income_ts[df_income_ts["Transaction"] == "Primary resources total"]
df_income_ts = df_income_ts.rename(columns={"Year": "year", "Value": "value"})
df_income_ts["indicator"] = "gdhi"
df_income_ts = df_income_ts[["local_authority", "year", "indicator", "value"]]

#children in low income families
df_children_ts = sheets["children"].copy()
df_children_ts["local_authority"] = df_children_ts["Local Authority"].apply(clean_la_name)
df_children_ts = df_children_ts.dropna(subset=["local_authority"])
df_children_ts["indicator"] = "children_low_income_" + df_children_ts["Type of Low Income Family"].str.lower()
df_children_ts = df_children_ts.rename(columns={"Year": "year", "Percentage": "value"})
df_children_ts = df_children_ts[["local_authority", "year", "indicator", "value"]]

# waste
df_waste_ts = sheets["waste"].copy()
df_waste_ts["local_authority"] = df_waste_ts["Local Authority"].apply(clean_la_name)
df_waste_ts["Value"] = pd.to_numeric(df_waste_ts["Value"], errors="coerce")
waste_indicator_map = {
    "Percentage of household waste sent for reuse, recycling or composting": "pct_waste_recycled",
    "Residual household waste per household (kg/household)": "residual_waste_per_household_kg",
}
df_waste_ts = df_waste_ts[df_waste_ts["Attribute"].isin(waste_indicator_map)].copy()
df_waste_ts["indicator"] = df_waste_ts["Attribute"].map(waste_indicator_map)
df_waste_ts = df_waste_ts.rename(columns={"Year": "year", "Value": "value"})
df_waste_ts = df_waste_ts[["local_authority", "year", "indicator", "value"]]

# life satisfaction/anxiety
df_life_ts = sheets["life"].copy()
df_life_ts["local_authority"] = df_life_ts["Local Authority"].apply(clean_la_name)
anxiety = df_life_ts[(df_life_ts["ScoreType"] == "Anxiety") & (df_life_ts["Score"] == "High (score 6 to 10) %")].copy()
anxiety["indicator"] = "anxiety_high_pct"
satisfaction = df_life_ts[(df_life_ts["ScoreType"] == "Life Satisfaction") & (df_life_ts["Score"] == "Low (score 0 to 4) %")].copy()
satisfaction["indicator"] = "life_satisfaction_low_pct"
df_life_ts = pd.concat([anxiety, satisfaction]).rename(columns={"Year": "year", "Value": "value"})
df_life_ts = df_life_ts[["local_authority", "year", "indicator", "value"]]

#Monthly counts, averaged up to one value per LA per year per count type
df_rough_ts = sheets["rough_sleeping"].copy()
df_rough_ts["local_authority"] = df_rough_ts["Local Authority"].apply(clean_la_name)
df_rough_ts["year"] = pd.to_datetime(df_rough_ts["Month"]).dt.year
df_rough_ts = (df_rough_ts.groupby(["local_authority", "year", "Time Range"])["Value"]
               .mean().reset_index())
df_rough_ts["indicator"] = df_rough_ts["Time Range"].map({
    "Single Night": "rough_sleeping_single_night",
    "Long Term": "rough_sleeping_longterm",
})
df_rough_ts = df_rough_ts.rename(columns={"Value": "value"})[["local_authority", "year", "indicator", "value"]]

#raw destination counts becomes % going on to apprenticeship or work
df_post1618_ts = sheets["post16_18"].copy()
df_post1618_ts["local_authority"] = df_post1618_ts["Local Authority"].apply(clean_la_name)
df_post1618_ts = df_post1618_ts[df_post1618_ts["Sex"] == "Total"].dropna(subset=["local_authority"])
p1618_wide = df_post1618_ts.pivot_table(index=["local_authority", "Academic Year"], columns="Attribute",
                                          values="Value", aggfunc="sum").reset_index()
for col in ["education", "apprenticeship", "work"]:
    if col not in p1618_wide.columns:
        p1618_wide[col] = 0

positive_destinations = p1618_wide["apprenticeship"] + p1618_wide["work"]
all_destinations = p1618_wide["education"] + positive_destinations
p1618_wide["value"] = positive_destinations / all_destinations * 100
p1618_wide["indicator"] = "post16_18_positive_destination_pct"
df_post1618_ts = p1618_wide.rename(columns={"Academic Year": "year"})[["local_authority", "year", "indicator", "value"]]
df_active_ts = sheets["active"].copy()
df_active_ts["local_authority"] = df_active_ts["Local Authority"].apply(clean_la_name)
df_active_ts = df_active_ts.dropna(subset=["local_authority"])
df_active_ts["year"] = df_active_ts["Date Range"].str.extract(r"(\d{2})$")[0].astype(float) + 2000
df_active_ts["indicator"] = "pct_adults_active_raw"
df_active_ts = df_active_ts.rename(columns={"Value": "value"})[["local_authority", "year", "indicator", "value"]]

#create 1 long table of tidied indicators
panel_long = pd.concat([
    df_income_ts, df_children_ts, df_waste_ts, df_life_ts, df_rough_ts, df_post1618_ts, df_active_ts,
], ignore_index=True)

panel_long = panel_long.dropna(subset=["local_authority"])

#drops rows that arent lancashire districts
panel_dropped = sorted(set(panel_long["local_authority"]) - CANONICAL_LAS)
if panel_dropped:
    print(f"  [filter - time series panel] dropping non-LA rows: {panel_dropped}")
panel_long = panel_long[panel_long["local_authority"].isin(CANONICAL_LAS)]

# force numberic dtypes
before = panel_long["value"].notna().sum()
panel_long["value"] = pd.to_numeric(panel_long["value"], errors="coerce")
after = panel_long["value"].notna().sum()
if after < before:
    print(f"  [numeric coercion] {before - after} values were non-numeric and became NaN")
